# illust-asset-forge

上から順に実行する。各セルで書き換えるのは、先頭の変数（キャラクターの ID・枚数・表情名・画風名）だけ。

## 全体の流れ

1. **準備**（セル 1・2）: リポジトリとモデルを用意する
2. **画風の見比べ**（任意）: 同じ顔を画風ごとに見る
3. **顔**: 作って、ボタンで採用する
4. **表情**: 範囲を赤枠で決めてから作り、ボタンで採用する
5. **全身**: 作って、ボタンで採用する
6. **まとめ**: zip にして Claude に渡す

## 1. 準備（リポジトリ・ライブラリ）

Drive をつなぎ、リポジトリを取り、ライブラリを入れる。Claude が設定を直して push した後は、**このセルだけ**を実行し直せば取り込める（モデルは読み込み直さない）。

In [ ]:
import os
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")

REPO = "/content/illust-asset-forge"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/akabee0161/illust-asset-forge.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)

!pip install -q -r {REPO}/requirements-colab.txt
!pip install -q -e {REPO} --no-deps

os.chdir(REPO)
# pip install -e は、動いているカーネルの import には効かない。src を直接足す
if os.path.join(REPO, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO, "src"))
# 実行し直したときに古いコードのままにならないよう、読み込み済みの artgen を捨てる
for name in [m for m in sys.modules if m == "artgen" or m.startswith("artgen.")]:
    del sys.modules[name]
from artgen import forge as F
from artgen import picker as P
if "pipe" in globals():
    forge = F.attach(pipe, ".")

## 2. モデルの読み込み

`pipe` が無いときだけ読み込む（2 回目以降は何もしない）。メモリ不足で止まったときだけ、`F.load_pipeline(low_memory=True)` にする（遅くなる）。

**終わったら** → 次へ。

In [ ]:
if "pipe" not in globals():
    pipe = F.load_pipeline()
forge = F.attach(pipe, ".")

## 3. 画風の見比べ（任意）

`CHAR` の顔を、画風ごとに 2 枚ずつ作って並べる。気に入った画風名を、後ろの `STYLE` に書く。

In [ ]:
CHAR = "ally05"
paths = forge.compare_styles(CHAR, seeds=None, n=2)
P.show_styles(paths)

## 4. 顔

`CHARS` に作るキャラクターを並べる。1 枚できるたびに表示される（1 枚 20 秒ほど）。調整中は `n=2` など少なくし、一気に作るときは `n=16` などにして、後から下の「候補を開き直す」で選ぶ。`STYLE = None` なら設定の画風。

できたら、出てくる画面で採用する顔のボタンを押す。

In [ ]:
CHARS = ["ally05"]
STYLE = None
paths = forge.generate(CHARS, "face", n=4, style=STYLE, on_image=F.show_one)
P.pick(forge, paths)

## 4b. 候補を開き直す

中断・切断の後や、後から選ぶときに使う。`run_id` に Drive の `runs` のフォルダ名を書けば、その run の候補を開く。`None` なら最新の run。

In [ ]:
P.pick(forge, run_id=None)

## 5. 表情の範囲

採用した顔に、目と口の範囲（赤枠）をスライダーで合わせて保存する。表情を作る前に、キャラクターごとに 1 回やる。

In [ ]:
CHAR = "ally05"
P.edit_box(forge, CHAR)

## 6. 表情

採用した顔と決めた範囲から作る（パスは書かない）。表情名は `character.yaml` の `expressions` のキー（`hurt`・`shout` など）。ほかの表情は `EXPRESSION` を書き換えて実行し直す。

In [ ]:
CHAR = "ally05"
EXPRESSION = "hurt"
paths = forge.inpaint(CHAR, EXPRESSION, n=4, on_image=F.show_one)
P.pick(forge, paths)

## 7. 全身（参考資料）

顔と同じく、作ってからボタンで採用する。見るのは、服の種類・色・持ち物と、髪の色・髪型が顔と合っているか。

In [ ]:
paths = forge.generate(CHARS, "fullbody", n=4, on_image=F.show_one)
P.pick(forge, paths)

## 8. まとめ

採用したものを zip にして、ダウンロードする。この zip を Claude とのチャットに添付する。

In [ ]:
z = forge.pack()
print(z)
try:
    from google.colab import files
    files.download(str(z))
except Exception as e:
    print(f"ダウンロードできなかった（Drive の packs から取る）: {e}")

## 試すときの書き方

- 足すタグ・消すタグ: `forge.generate(CHARS, "face", extra="scar on cheek", remove="stubble")`（ネガティブは `extra_negative=`・`remove_negative=`）。`remove` は設定のタグにだけ効く。`extra` に設定と同じタグを重み付きで書くと、設定のタグが置き換わる（例: `extra="(stubble:1.3)"`）
- シードを固定: `seeds=[123, 456]`（渡すと `n` は無視される）
- 画風: `style="<画風名>"`
- guidance・steps（inpaint は `strength` も）: `guidance=6`、`steps=30`
- 選ぶ画面を使わずに採用する: `forge.adopt("<ファイル名>", flip=True)`（`flip=True` で左右反転して採用）
- 画像を並べて見る: `F.show(paths)`
- **YAML は Colab で書き換えない**（次の pull でぶつかる）。上書きの内容は PNG の条件（`overrides`）に残るので、Claude が設定に戻す。